In [2]:
%pip install -r requirements.txt

^C
Note: you may need to restart the kernel to use updated packages.


<small>

# Performing Vector Similarity Search in Azure SQL Database

In this guide, we demonstrate how to query an embedding table in Azure SQL Database to retrieve the most relevant customer reviews based on a user's search query.

## Overview

When a user enters a search query, we convert that text into its **vector representation** using embeddings. This vector can then be compared to the vectors of existing customer reviews stored in the database.

By calculating the **cosine similarity** between the query vector and each review vector, we can determine which reviews are most relevant to the user's intent. These are the reviews most likely associated with the product or experience the user is looking for.

The most relevant reviews—those with the highest similarity—help users discover products or experiences that match their interests.

## Vector Distance in SQL

Azure SQL DB provides built-in support for calculating vector distances. The function syntax is:

```sql
VECTOR_DISTANCE('distance metric', vector1, vector2)


In [ ]:
import os
import json
from dotenv import load_dotenv
import pyodbc
import struct
from azure.identity import DefaultAzureCredential

def vector_search_sql(query, num_results=5):
    # Load environment variables from .env file
    load_dotenv()

    #Use the get_mssql_connection function to get the connection string details
    conn = get_mssql_connection()

    # Create a cursor object
    cursor = conn.cursor()

    # Generate the query embedding for the user's search query
    user_query_embedding = get_embedding(query)

    # Convert user_query_embedding to a JSON string
    user_query_embedding_json = json.dumps(user_query_embedding)

    # SQL query for similarity search using the function vector_distance to calculate cosine similarity
    sql_similarity_search = """
    SELECT TOP(?) ProductId, Summary, text,
           1 - vector_distance('cosine', cast(? as vector(1536)), [vector]) AS similarity_score
    FROM dbo.embeddings
    ORDER BY similarity_score desc
    """

    cursor.execute(sql_similarity_search, (num_results, user_query_embedding_json))
    results = cursor.fetchall()

    # Close the database connection
    conn.close()

    return results


In [ ]:
# Assuming you have implemented the `vector_search_sql` function as shown earlier

# Example usage
query = "oatmeal options for my toddler"
num_results = 4
search_results = vector_search_sql(query, num_results)
for result in search_results:
    product_id = result[0]  # Assuming ProductId is the first column
    summary = result [1]
    text = result [2]
    similarity_score = result[3]  # Assuming similarity_score is the third column
    
    print(f"Product ID: {product_id}")
    print(f"summary : {summary}")
    print (f"Text : {text}")
    print(f"Similarity Score: {similarity_score}\n")


Connecting using SQL Authentication
Product ID: B001DIM8K8
summary : Nothing like the crappy stuff I grew up on.
Text : My daughter and I eat low-carb. Which oatmeal is not really, by the way. But I was looking for something that we could eat in the morning that was not meat/dairy/egg, and was gluten-free, and filling, without being pure sugar like most breakfast foods and especially grains. I decided to try these long-cooking irish oats. We use 2 cups water and 1/2 cup dry oats, for 1/4cup dry each. It takes this stuff 30 minutes+ to cook so it is not fast food, for sure. But strangely enough, it's filling. Adding fats (couple tablespoons of cream) to the serving helps that of course. In any case, it is much tastier, and much more long-lasting for keeping us satisfied, than I expected, and surprisingly stable on blood sugar. Nor has it set off cravings like most grains, fruits, etc. do if we eat those; the sugars digest so slowly in this apparently. It is more solid and nutty than the

<small>

# Using Embeddings from a Vector Database to Augment LLM Generation

In this section, we’ll define a helper function that feeds prompts into the [Completions model](https://learn.microsoft.com/en-us/azure/ai-services/openai/concepts/models#gpt-35-models) and establishes an interactive loop where you can ask questions and receive responses grounded in your data.

## Overview

The `generate_completion` function is designed to interact with the GPT-3.5 model by providing it with both prompts and system-level instructions.

It leverages the results from the previously defined `vector_search_sql` function to supply relevant context to the language model. This helps generate more accurate, data-grounded responses.

We are using the `gpt-35-turbo-16k` model in this example.

## Learn More

For additional details on working with Azure OpenAI GPT chat models, refer to the [official quickstart guide](https://learn.microsoft.com/en-us/azure/ai-services/openai/chatgpt-quickstart?tabs=command-line%2Cpython-new&pivots=programming-language-python).

</small>


In [26]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI

# Load environment variables from .env file
load_dotenv()

# Retrieve the API key and endpoint from the environment variables
api_key = os.getenv('AZURE_OPENAI_API_KEY')
azure_endpoint = os.getenv('AZURE_OPENAI_ENDPOINT')

# Create a chat completion request
client = AzureOpenAI(
    api_key=api_key,
    api_version="2023-05-15",
    azure_endpoint=azure_endpoint
)


def generate_completion(search_results, user_input):
    system_prompt = '''
You are an intelligent & funny assistant who will exclusively answer based on the data provided in the `search_results`:
- Use the information from `search_results` to generate your responses. If the data is not a perfect match for the user's query, use your best judgment to provide helpful suggestions and include the following format:
  Product ID: {product_id}
  Summary: {summary}
  Review: {text}
  Similarity Score: {similarity_score}
- Avoid any other external data sources.
- Add a fun fact related to the overall product searched at the end of the recommendations.
'''

    messages = [{"role": "system", "content": system_prompt}]
    search_results = vector_search_sql(user_input, num_results)
    
    # Create an empty list to store the results
    result_list = []

    # Iterate through the search results and append relevant information to the list
    for result in search_results:
        product_id = result[0]  # Assuming ProductId is the first column
        summary = result[1]
        text = result[2]
        similarity_score = result[3]  # Assuming similarity_score is the third column
        
        # Append the relevant information as a dictionary to the result_list
        result_list.append({
            "product_id": product_id,
            "summary": summary,
            "text": text,
            "similarity_score": similarity_score
        })

    #print (result_list)
    messages.append({"role": "system", "content": f"{result_list}"})
    messages.append({"role": "user", "content": user_input})
    response = client.chat.completions.create(model='chatcompletion', messages=messages, temperature=0) #replace with your model deployment name

    return response.dict()

In [27]:
# Create a loop of user input and model output to perform Q&A on the FineFoods Sample data

print("*** What products are you looking for? Ask me & I can help you :) Type 'end' to end the session.\n")

while True:
    user_input = input("User prompt: ")
    if user_input.lower() == "end":
        break

    # Print the user's question
    print(f"\nUser asked: {user_input}")

    # Assuming vector_search_sql and generate_completion are defined functions that work correctly
    search_results = vector_search_sql(user_input)
    completions_results = generate_completion(search_results, user_input)

    # Print the model's response
    print("\nAI's response:")
    print(completions_results['choices'][0]['message']['content'])

# The loop will continue until the user types 'end'


*** What products are you looking for? Ask me & I can help you :) Type 'end' to end the session.




User asked: I want some healthy options for cat food. Mind you my cat is super fussy and doesnt like stuff easily



AI's response:
I understand that your cat is a picky eater, but I have some healthy options for you to consider based on the reviews:

1. Product ID: B002PNGY28
   Summary: Great soft little treats for your finicky feline
   Review: I have 2 cats. One will eat anything. The other one is soooo picky. She's had to have some of her teeth removed due to a mouth virus, so I needed to find some good soft treats. She likes these more than anything I've tried. They are really thin little tiny strips. I like Wellness, all natural products. Both cats like their brand cat food as well.
   Similarity Score: 0.651522667741025

2. Product ID: B003SE19UK
   Summary: Palatable and healthy
   Review: Before I was educated about feline nutrition, I allowed my cats to become addicted to dry cat food. I always offered both canned and dry, but wish I would have fed them premium quality canned food and limited dry food. I have two 15-year-old cats and two 5-year-old cats. The only good quality dry foods th

You can use also use Semantic Search with Keyword search in SQL DB along with using filters and aggregations . Check out the TSQL in the same repo